# Phân cụm giao dịch IBFT bằng Gaussian Mixture Model (GMM)

Notebook chạy **GMM** trên 11 đặc trưng đã chuẩn bị ở notebook 01.
Một điểm dữ liệu là một giao dịch; số tài khoản không đi vào mô hình.

**Quy trình:** fit trên train → chọn K và seed bằng validation → báo cáo cuối trên test.
Nhãn cụm là thành phần có xác suất hậu nghiệm cao nhất (`predict`).

**Cách chạy:** chọn kernel `.venv`, rồi *Restart Kernel → Run All*.
Đọc `output/prepared/`, ghi kết quả vào `output/gmm/` theo đường dẫn đã thống nhất.
Chạy lại cập nhật kết quả tại đây; `RUN_ID` chỉ là mã nhận diện trong cấu hình.

## 1. Cấu hình

Mọi con số có thể chỉnh nằm ở ô này. Giữ **giống hệt** ở notebook K-Means và Autoencoder + K-Means
để notebook 02 so sánh công bằng.

In [ ]:
RUN_ID = "run_001"
K_GRID = list(range(3, 11))          # các giá trị K cần quét
SEEDS = [42, 43, 44, 45, 46]         # các random_state

GMM_PARAMS = dict(covariance_type="full", n_init=3, max_iter=300,
                  tol=1e-3, reg_covar=1e-6, init_params="kmeans")
SILHOUETTE_TRAIN_SIZE = 10_000       # validation/test dùng toàn bộ
PCA_SAMPLE_SIZE = 5_000              # số điểm test vẽ trên hình PCA
EVAL_SEED = 42                       # seed chọn mẫu đánh giá và vẽ hình

## 2. Thư viện và đường dẫn

In [ ]:
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.mixture import GaussianMixture
from sklearn.exceptions import ConvergenceWarning
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score


def find_project_root():
    """Đi ngược lên các thư mục cha để tìm thư mục dự án (nơi có output/prepared)."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "output" / "prepared" / "model_input.parquet").is_file():
            return folder
    raise FileNotFoundError("Chưa có model_input.parquet; hãy chạy notebook 01 trước.")


PROJECT_ROOT = find_project_root()
PREPARED_DIR = PROJECT_ROOT / "output" / "prepared"
RUN_DIR = PROJECT_ROOT / "output"
METHOD_DIR = RUN_DIR / "gmm"
FIGURES_DIR = METHOD_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def save_figure(fig, filename):
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})
print("Dữ liệu:", PREPARED_DIR)
print("Kết quả:", METHOD_DIR)

## 3. Đọc dữ liệu dùng chung

Giữ nguyên đặc trưng và cách chia tập từ notebook 01. Không điền thiếu, chuẩn hóa
hay tính lại lịch sử ở đây. Các `assert` chỉ kiểm tra những điều cốt lõi: đủ 11 cột,
không thiếu giá trị, và train → validation → test nối tiếp nhau theo thời gian.

In [ ]:
RAW_FEATURES = [
    "log_amount", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
    "log_sender_recency_days", "log_sender_tx_count_7d", "log_pair_tx_count_7d",
    "receiver_share_7d", "is_first_sender_tx", "is_new_receiver",
]
FEATURE_COLUMNS = [f"x_{name}" for name in RAW_FEATURES]
SPLITS = ["train", "validation", "test"]

model_input = pd.read_parquet(PREPARED_DIR / "model_input.parquet").reset_index(drop=True)

# Kiểm tra cốt lõi
assert all(col in model_input.columns for col in FEATURE_COLUMNS), "Thiếu cột đặc trưng x_*"
assert np.isfinite(model_input[FEATURE_COLUMNS].to_numpy()).all(), "Đặc trưng có giá trị thiếu hoặc vô hạn"
assert model_input["transaction_id"].is_unique, "transaction_id bị trùng"
assert set(model_input["split"]) == set(SPLITS), "split phải gồm train/validation/test"

split_summary = (
    model_input.groupby("split")
    .agg(row_count=("transaction_id", "size"),
         start_time=("transaction_datetime", "min"),
         end_time=("transaction_datetime", "max"))
    .reindex(SPLITS)
)
assert split_summary.loc["train", "end_time"] < split_summary.loc["validation", "start_time"]
assert split_summary.loc["validation", "end_time"] < split_summary.loc["test", "start_time"]

# Ma trận đặc trưng của từng tập
split_masks = {s: model_input["split"].eq(s).to_numpy() for s in SPLITS}
X = {s: model_input.loc[split_masks[s], FEATURE_COLUMNS].to_numpy(dtype=np.float64) for s in SPLITS}

display(split_summary)
print(f"{len(model_input):,} giao dịch; {len(FEATURE_COLUMNS)} đặc trưng.")

## 4. Hàm tính ba chỉ số

- **Silhouette:** cao hơn thì các cụm tách biệt và gọn hơn (khoảng cách Euclid).
- **Davies–Bouldin:** thấp hơn tốt hơn.
- **Calinski–Harabasz:** cao hơn tốt hơn khi so sánh cùng K; không dùng để chọn K.

Các chỉ số luôn tính trong không gian 11 đặc trưng `x_*`. Silhouette trên train dùng một
mẫu cố định (chọn một lần, dùng lại cho mọi K và seed) vì tính đủ rất chậm.

In [ ]:
rng = np.random.RandomState(EVAL_SEED)
silhouette_rows = {
    "train": rng.permutation(len(X["train"]))[:SILHOUETTE_TRAIN_SIZE],
    "validation": np.arange(len(X["validation"])),
    "test": np.arange(len(X["test"])),
}


def evaluate(split, labels):
    """Tính ba chỉ số cho một tập, trong không gian x_* gốc."""
    values = X[split]
    rows = silhouette_rows[split]
    scores = dict(n_eval=len(values), silhouette=None, davies_bouldin=None, calinski_harabasz=None)
    if not 2 <= np.unique(labels).size < len(values):
        return scores  # Các chỉ số không xác định nếu chỉ có một cụm.
    scores["davies_bouldin"] = float(davies_bouldin_score(values, labels))
    scores["calinski_harabasz"] = float(calinski_harabasz_score(values, labels))
    if 2 <= np.unique(labels[rows]).size < len(rows):
        scores["silhouette"] = float(silhouette_score(values[rows], labels[rows]))
    return scores


display(pd.DataFrame({
    "split": SPLITS,
    "n_eval": [len(X[s]) for s in SPLITS],
    "silhouette_n_eval": [len(silhouette_rows[s]) for s in SPLITS],
}))

### 4.1. Hàm fit GMM và xử lý khi không hội tụ

Mỗi cặp (K, seed) thử `full` trước. Nếu không hội tụ, covariance bị lỗi hoặc likelihood
không hữu hạn thì thử lại **cùng K, seed** với `diag`. Nếu cả hai đều thất bại,
notebook dừng và báo lý do. Lỗi đầu vào khác được báo trực tiếp.

Lưu một dòng thông tin cho mỗi lượt fit: covariance thực tế, số vòng EM,
lý do chuyển `diag` (nếu có) và thời gian fit. Thông tin này nằm trong `config.json`.

In [ ]:
def fit_gmm(k, seed):
    """Fit chỉ trên train; thử full rồi diag khi gặp vấn đề hội tụ/covariance."""
    start = perf_counter()
    fallback_reason = None
    for covariance in ["full", "diag"]:
        params = {**GMM_PARAMS, "covariance_type": covariance}
        model = GaussianMixture(n_components=k, random_state=seed, **params)
        problem = None
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                model.fit(X["train"])
            if not model.converged_:
                problem = "Không hội tụ trong giới hạn max_iter."
            elif not np.isfinite(model.score(X["train"])):
                problem = "Log-likelihood không hữu hạn."
        except np.linalg.LinAlgError as error:
            problem = str(error)
        except ValueError as error:
            if not any(word in str(error).lower() for word in ["covariance", "positive definite", "ill-defined"]):
                raise
            problem = str(error)

        if problem is None:
            info = {
                "k": k, "seed": seed, "covariance_type": covariance,
                "n_iter": int(model.n_iter_), "fallback_reason": fallback_reason,
                "fit_seconds": float(perf_counter() - start),
            }
            return model, info
        if covariance == "full":
            fallback_reason = problem
            print(f"K={k}, seed={seed}: chuyển sang diag. Lý do: {problem}")
    raise RuntimeError(f"K={k}, seed={seed}: full thất bại ({fallback_reason}); diag thất bại ({problem}).")

## 5. Quét K và seed

Mỗi cặp (K, seed) fit một GMM **chỉ trên train**; validation và test chỉ được `predict`.
Các chỉ số phân cụm tính từ nhãn cứng trong không gian 11 đặc trưng gốc.
Với 8 K × 5 seed có **40 lượt fit** và **120 dòng kết quả**.

BIC/AIC là chỉ số tham khảo, không dùng để chọn K. `inertia` và `recon_mse` để trống.
`fit_seconds` bao gồm lần thử `full` và lần thử `diag` nếu cần; lặp lại ở ba dòng của cùng mô hình.

In [ ]:
rows = []
fit_log = []
start = perf_counter()

for k in K_GRID:
    for seed in SEEDS:
        model, info = fit_gmm(k, seed)
        fit_log.append(info)
        for split in SPLITS:
            labels_split = model.predict(X[split])
            rows.append({
                "method": "gmm", "k": k, "seed": seed, "split": split,
                **evaluate(split, labels_split),
                "fit_seconds": info["fit_seconds"], "inertia": None,
                "bic": float(model.bic(X[split])), "aic": float(model.aic(X[split])),
                "recon_mse": None,
            })
    print(f"Xong K={k}", flush=True)

SWEEP_COLUMNS = [
    "method", "k", "seed", "split", "n_eval",
    "silhouette", "davies_bouldin", "calinski_harabasz", "fit_seconds",
    "inertia", "bic", "aic", "recon_mse",
]
k_sweep = pd.DataFrame(rows, columns=SWEEP_COLUMNS)
assert len(k_sweep) == len(K_GRID) * len(SEEDS) * len(SPLITS)
k_sweep.to_csv(METHOD_DIR / "k_sweep.csv", index=False, encoding="utf-8-sig")

fallbacks = pd.DataFrame(fit_log).query("covariance_type == 'diag'")
print(f"Hoàn tất {len(fit_log)} lượt fit trong {perf_counter() - start:.1f} giây; {len(fallbacks)} lượt dùng diag.")
if not fallbacks.empty:
    display(fallbacks)
display(k_sweep.head(6).round(4))

## 6. Chọn K và seed bằng validation

Quy tắc (không đọc chỉ số test):

1. **K tốt nhất** là K có Silhouette validation trung bình (qua các seed) cao nhất.
2. **Seed tốt nhất** là seed có Silhouette validation cao nhất trong K đó.
3. Nếu bằng điểm thì chọn K nhỏ hơn / seed nhỏ hơn.

In [ ]:
validation_scores = k_sweep[k_sweep["split"] == "validation"]
if validation_scores["silhouette"].isna().any():
    raise ValueError("Có Silhouette validation không xác định; hãy kiểm tra nhãn cụm trước khi chọn K.")

selection_table = (
    validation_scores.groupby("k")["silhouette"]
    .agg(mean="mean", std="std")
    .fillna(0.0)
    .reset_index()
    .sort_values(["mean", "k"], ascending=[False, True])
    .reset_index(drop=True)
)
best_k = int(selection_table.loc[0, "k"])
best_validation_mean = float(selection_table.loc[0, "mean"])
best_validation_std = float(selection_table.loc[0, "std"])

best_seed = int(
    validation_scores[validation_scores["k"] == best_k]
    .sort_values(["silhouette", "seed"], ascending=[False, True])
    .iloc[0]["seed"]
)

display(selection_table.rename(columns={
    "mean": "validation_silhouette_mean", "std": "validation_silhouette_std",
}).round(6))
print(f"Chọn K={best_k}, seed={best_seed}.")
print(f"Silhouette validation trung bình: {best_validation_mean:.6f} ± {best_validation_std:.6f}.")
if best_k in (min(K_GRID), max(K_GRID)):
    print("Lưu ý: K được chọn nằm ở biên của lưới K; giá trị tốt hơn có thể nằm ngoài lưới.")

## 7. BIC/AIC và ba chỉ số theo K

BIC/AIC dùng **train**; ba chỉ số phân cụm dùng **validation**. Đường là trung bình,
dải là ±1 độ lệch chuẩn giữa các seed, không phải khoảng tin cậy.
Đường dọc đánh dấu K đã chọn bằng Silhouette validation, không phải BIC/AIC.
Nếu có lượt chuyển `diag`, biểu đồ dùng đúng kết quả sau chuyển đổi đã ghi ở mục 5.

In [ ]:
def plot_mean_band(ax, data, column, color):
    """Vẽ trung bình theo K và dải ± 1 độ lệch chuẩn giữa các seed."""
    summary = data.groupby("k")[column].agg(["mean", "std"]).reindex(K_GRID)
    summary["std"] = summary["std"].fillna(0.0)
    ax.plot(K_GRID, summary["mean"], marker="o", color=color, linewidth=2)
    ax.fill_between(K_GRID, summary["mean"] - summary["std"], summary["mean"] + summary["std"],
                    color=color, alpha=0.15)
    ax.axvline(best_k, color="#555555", linestyle="--", linewidth=1)
    ax.set_xticks(K_GRID)
    ax.grid(axis="y", alpha=0.25)


train_scores = k_sweep[k_sweep["split"] == "train"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, color in zip(axes, ["bic", "aic"], ["#2457A6", "#B27619"]):
    plot_mean_band(ax, train_scores, metric, color)
    ax.set(title=f"{metric.upper()} trên train (tham khảo)", xlabel="Số cụm K", ylabel=metric.upper())
fig.suptitle(f"GMM — BIC/AIC trung bình ± độ lệch chuẩn của {len(SEEDS)} seed; K được chọn = {best_k}")
fig.tight_layout()
save_figure(fig, "bic_aic_by_k.png")

In [ ]:
metric_specs = [
    ("silhouette", "Silhouette (cao hơn tốt hơn)", "#2457A6"),
    ("davies_bouldin", "Davies–Bouldin (thấp hơn tốt hơn)", "#B27619"),
    ("calinski_harabasz", "Calinski–Harabasz (tham khảo)", "#8B4C70"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (metric, title, color) in zip(axes, metric_specs):
    plot_mean_band(ax, validation_scores, metric, color)
    ax.set(title=title, xlabel="Số cụm K", ylabel="Giá trị chỉ số")
fig.suptitle(f"GMM trên validation — trung bình ± độ lệch chuẩn của {len(SEEDS)} seed")
fig.tight_layout()
save_figure(fig, "validation_metrics_by_k.png")

## 8. Huấn luyện lại mô hình cuối và báo cáo test

Fit lại đúng (K, seed) đã chọn, **vẫn chỉ trên train** (không gộp validation vào train).
Gán nhãn cả ba tập và tính chỉ số. Kết quả cuối được báo cáo trên **test**.

In [ ]:
final_model, final_fit_info = fit_gmm(best_k, best_seed)
final_fit_seconds = final_fit_info["fit_seconds"]

final_labels = {s: final_model.predict(X[s]) for s in SPLITS}
final_scores = {s: evaluate(s, final_labels[s]) for s in SPLITS}

# Bảng nhãn cho cả ba tập (dùng để lưu và tra cứu theo cụm)
labels = model_input[["transaction_id", "split"]].copy()
labels["cluster"] = -1
for s in SPLITS:
    labels.loc[split_masks[s], "cluster"] = final_labels[s]
assert labels["cluster"].between(0, best_k - 1).all()

final_score_table = pd.DataFrame.from_dict(final_scores, orient="index").reindex(SPLITS)
final_score_table.index.name = "split"
display(final_score_table.round(6))
display(Markdown(
    f"**Kết quả cuối trên test:** K={best_k}, seed={best_seed}, covariance={final_model.covariance_type}; "
    f"Silhouette **{final_scores['test']['silhouette']:.4f}**, "
    f"Davies–Bouldin **{final_scores['test']['davies_bouldin']:.4f}**, "
    f"Calinski–Harabasz **{final_scores['test']['calinski_harabasz']:.2f}**."
))

## 9. Mô tả cụm trên tập test

Bảng mô tả dùng nhãn cuối trên **test**, ghép theo `transaction_id` với
`features_unscaled.parquet` và `transaction_lookup.parquet`.
Số tiền tính bằng VND gốc; trung vị bỏ qua số tiền thiếu và ghi kèm số dòng thiếu.
Số giao dịch 7 ngày được khôi phục từ `log_sender_tx_count_7d` bằng `expm1`.

Các cờ «mới» và «đầu tiên» chỉ nói về lịch sử quan sát được. Số thứ tự cụm chỉ là mã định danh,
không thể hiện tốt hay xấu.

In [ ]:
profile_features = pd.read_parquet(
    PREPARED_DIR / "features_unscaled.parquet",
    columns=["transaction_id", "split", "is_new_receiver", "is_first_sender_tx", "log_sender_tx_count_7d"],
)
lookup = pd.read_parquet(
    PREPARED_DIR / "transaction_lookup.parquet",
    columns=["transaction_id", "transaction_datetime", "transactionamount"],
)

profile_data = (
    labels[labels["split"] == "test"]
    .merge(profile_features, on=["transaction_id", "split"], how="left", validate="one_to_one")
    .merge(lookup, on="transaction_id", how="left", validate="one_to_one")
)
assert len(profile_data) == len(X["test"])
assert profile_data["transaction_datetime"].notna().all(), "Không ghép được thông tin giao dịch"

profile_data["hour"] = profile_data["transaction_datetime"].dt.hour
profile_data["sender_tx_count_7d"] = np.rint(np.expm1(profile_data["log_sender_tx_count_7d"]))

cluster_profile = (
    profile_data.groupby("cluster")
    .agg(
        transaction_count=("transaction_id", "size"),
        median_amount_vnd=("transactionamount", "median"),
        modal_hour=("hour", lambda v: int(v.mode().iloc[0])),   # giờ phổ biến nhất (bằng nhau thì lấy giờ nhỏ hơn)
        is_new_receiver_rate=("is_new_receiver", "mean"),
        is_first_sender_tx_rate=("is_first_sender_tx", "mean"),
        median_sender_tx_count_7d=("sender_tx_count_7d", "median"),
        amount_missing_count=("transactionamount", lambda v: int(v.isna().sum())),
    )
    .reindex(range(best_k))
)
cluster_profile[["transaction_count", "amount_missing_count"]] = (
    cluster_profile[["transaction_count", "amount_missing_count"]].fillna(0).astype(int)
)
cluster_profile.insert(1, "transaction_share", cluster_profile["transaction_count"] / len(profile_data))
cluster_profile = cluster_profile.reset_index()

display(cluster_profile.style.format({
    "transaction_count": "{:,.0f}", "transaction_share": "{:.2%}",
    "median_amount_vnd": "{:,.0f}", "modal_hour": "{:.0f}",
    "is_new_receiver_rate": "{:.2%}", "is_first_sender_tx_rate": "{:.2%}",
    "median_sender_tx_count_7d": "{:.1f}", "amount_missing_count": "{:.0f}",
}, na_rep="—"))

## 10. Kích thước cụm

Biểu đồ thể hiện số giao dịch test trong từng cụm. Nhãn phía trên ghi cả số lượng
và tỷ lệ trên toàn bộ test, để nhận biết các cụm rất nhỏ hoặc chiếm ưu thế.

In [ ]:
cluster_colors = {c: plt.get_cmap("tab10")(c) for c in range(best_k)}

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(cluster_profile["cluster"], cluster_profile["transaction_count"],
       color=[cluster_colors[int(c)] for c in cluster_profile["cluster"]], width=0.7)
for row in cluster_profile.itertuples():
    ax.text(row.cluster, row.transaction_count,
            f"{row.transaction_count:,}\n({row.transaction_share:.1%})",
            ha="center", va="bottom", fontsize=10)
ax.set(title=f"Kích thước cụm GMM trên {len(X['test']):,} giao dịch test",
       xlabel="Mã cụm", ylabel="Số giao dịch",
       xticks=list(range(best_k)),
       ylim=(0, max(1, cluster_profile["transaction_count"].max()) * 1.22))
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
fig.tight_layout()
save_figure(fig, "cluster_sizes_test.png")

## 11. Hình PCA hai chiều

PCA chỉ phục vụ hiển thị: fit PCA trên train rồi chiếu một mẫu tối đa 5.000 giao dịch test.
Màu là nhãn GMM đã gán trong không gian **11 chiều**; không phân cụm lại trên hai tọa độ PCA.
Các điểm chồng nhau trong hình chưa đủ để kết luận các cụm trùng nhau trong toàn bộ không gian đặc trưng.

In [ ]:
pca = PCA(n_components=2, svd_solver="full")
pca.fit(X["train"])

plot_rows = np.random.RandomState(EVAL_SEED).permutation(len(X["test"]))[:PCA_SAMPLE_SIZE]
projected = pca.transform(X["test"][plot_rows])
plot_labels = final_labels["test"][plot_rows]

fig, ax = plt.subplots(figsize=(10, 6))
for cluster in range(best_k):
    selected = plot_labels == cluster
    ax.scatter(projected[selected, 0], projected[selected, 1],
               color=cluster_colors[cluster], s=12, alpha=0.5, linewidths=0, label=f"Cụm {cluster}")
ax.set(title=f"PCA của {len(plot_rows):,} giao dịch test — màu theo cụm GMM",
       xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} phương sai train)",
       ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%} phương sai train)")
ax.legend(title="Nhãn cụm", frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, "pca_test.png")
print(f"Hai thành phần PCA giữ {pca.explained_variance_ratio_.sum():.2%} phương sai trên train.")

## 12. Độ chắc chắn gán cụm trên test

Lấy xác suất hậu nghiệm lớn nhất từ `predict_proba` cho mỗi giao dịch.
Giá trị cao thể hiện mô hình nghiêng mạnh về một thành phần; **không phải xác suất
phân cụm đúng**, vì không có nhãn chuẩn. Histogram và bảng tóm tắt dùng toàn bộ test.

In [ ]:
posterior = final_model.predict_proba(X["test"])
confidence = posterior.max(axis=1)
confidence_summary = {
    "minimum": float(confidence.min()),
    "p10": float(np.quantile(confidence, 0.10)),
    "median": float(np.median(confidence)),
    "p90": float(np.quantile(confidence, 0.90)),
    "maximum": float(confidence.max()),
}

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(confidence, bins=np.linspace(0, 1, 21), color="#2457A6", edgecolor="white")
ax.axvline(confidence_summary["median"], color="#555555", linestyle="--",
           label=f"Trung vị = {confidence_summary['median']:.3f}")
ax.set(title=f"GMM: xác suất hậu nghiệm lớn nhất trên {len(X['test']):,} giao dịch test",
       xlabel="Xác suất hậu nghiệm lớn nhất", ylabel="Số giao dịch", xlim=(0, 1))
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "assignment_confidence_test.png")
display(pd.Series(confidence_summary, name="max_posterior").to_frame().round(6))

## 13. Nhận xét và giới hạn

Đọc các chỉ số cùng bảng mô tả cụm; không gán ý nghĩa nghiệp vụ chỉ từ số thứ tự cụm.
Các gạch đầu dòng dưới đây **tự sinh từ kết quả của lần chạy này**, nên không bị lỗi thời khi chạy lại.

In [ ]:
largest = cluster_profile.loc[cluster_profile["transaction_count"].idxmax()]
most_active = cluster_profile.loc[cluster_profile["median_sender_tx_count_7d"].idxmax()]
highest_amount = cluster_profile.loc[cluster_profile["median_amount_vnd"].idxmax()]

display(Markdown(
    f"- K được chọn là **{best_k}** (seed {best_seed}), covariance **{final_model.covariance_type}**.\n"
    f"- Có **{len(fallbacks)}/{len(fit_log)}** lượt quét cần chuyển sang diag.\n"
    f"- Cụm **{int(largest['cluster'])}** lớn nhất trên test, chiếm **{largest['transaction_share']:.1%}** giao dịch.\n"
    f"- Cụm **{int(most_active['cluster'])}** có trung vị số giao dịch 7 ngày của người gửi cao nhất: "
    f"**{most_active['median_sender_tx_count_7d']:.1f}**.\n"
    f"- Cụm **{int(highest_amount['cluster'])}** có số tiền trung vị cao nhất: "
    f"**{highest_amount['median_amount_vnd']:,.0f} VND**."
))

**Giới hạn cần nhớ.** Dữ liệu là giả lập, không có nhãn đúng và không phục vụ phát hiện gian lận.
GMM mô tả hỗn hợp Gauss; `full` cho phép tương quan giữa các đặc trưng, `diag` chỉ giữ
phương sai riêng. Hai cờ nhị phân không phải biến Gauss liên tục nên cần thận trọng
khi diễn giải likelihood và độ chắc chắn gán cụm. Ba chỉ số tính trên nhãn cứng thiên
về cụm gọn theo khoảng cách Euclid, không đo đầy đủ chất lượng mô hình mật độ.
BIC/AIC chỉ tham khảo; Calinski–Harabasz chỉ so sánh giữa các phương pháp ở cùng K.
Notebook này chưa kết luận GMM tốt hơn K-Means hay Autoencoder.

## 14. Lưu kết quả theo định dạng chung

Lưu `config.json`, `k_sweep.csv` (đã lưu ở mục 5), `metrics.json`, `labels.parquet`,
`cluster_profile.csv`, `model.joblib` và 5 hình trong `figures/`.
Ghi cấu hình chung vào `output/config.json`. Không tạo thêm file chẩn đoán:
thông tin fit và lý do chuyển `diag` nằm trong cấu hình riêng của GMM.

In [ ]:
cluster_sizes = {
    s: {str(c): int(np.sum(final_labels[s] == c)) for c in range(best_k)} for s in SPLITS
}

# Cấu hình chung của lần chạy (ba phương pháp dùng cùng nội dung này)
run_config = {
    "run_id": RUN_ID,
    "k_grid": K_GRID,
    "seeds": SEEDS,
    "feature_columns": FEATURE_COLUMNS,
    "input_file": "output/prepared/model_input.parquet",
    "split_row_counts": {s: len(X[s]) for s in SPLITS},
    "metric_protocol": {
        "space": "prepared_x_features",
        "silhouette_train_sample_size": SILHOUETTE_TRAIN_SIZE,
        "silhouette_sample_seed": EVAL_SEED,
        "silhouette_validation_and_test": "all_rows",
        "davies_bouldin_and_calinski_harabasz": "all_rows_per_split",
    },
    "selection_rule": "K = highest mean validation silhouette over seeds; seed = best validation silhouette within K; test not used",
}
write_json(RUN_DIR / "config.json", run_config)

# Cấu hình riêng của GMM
write_json(METHOD_DIR / "config.json", {
    **run_config,
    "method": "gmm",
    "hyperparameters": GMM_PARAMS,
    "fit_diagnostics": fit_log,
    "final_fit_diagnostics": final_fit_info,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
})

# Chỉ số cuối
write_json(METHOD_DIR / "metrics.json", {
    "method": "gmm",
    "run_id": RUN_ID,
    "best_k": best_k,
    "best_seed": best_seed,
    "best_validation_mean_silhouette": best_validation_mean,
    "best_validation_std_silhouette": best_validation_std,
    "metrics": final_scores,
    "non_empty_clusters": {s: int(np.unique(final_labels[s]).size) for s in SPLITS},
    "cluster_sizes": cluster_sizes,
    "final_fit_seconds": final_fit_seconds,
    "covariance_type": final_model.covariance_type,
    "converged": bool(final_model.converged_),
    "confidence_test": confidence_summary,
    "cluster_profile_split": "test",
    "pca_explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
})

labels.to_parquet(METHOD_DIR / "labels.parquet", index=False)
cluster_profile.to_csv(METHOD_DIR / "cluster_profile.csv", index=False, encoding="utf-8-sig")
joblib.dump(final_model, METHOD_DIR / "model.joblib")

# Kiểm tra nhanh: mô hình lưu ra đọc lại cho đúng nhãn test
saved_model = joblib.load(METHOD_DIR / "model.joblib")
assert (saved_model.predict(X["test"]) == final_labels["test"]).all()

print("Đã lưu vào:", METHOD_DIR)
for path in sorted(METHOD_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(METHOD_DIR))